In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
test = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/Book/0119-similar_pr/SVD-0119/result/SVD_Single_Injection/5/f_Adult_500u_pos5_neg1_all_25recommendation.csv")

In [3]:
test.head(50)

,user_id,book_id,est_score,rank,genre_g1,genre_g2,genres_all
0,1,9534,5.792399,1,Unknown,NaN,Unknown
1,1,7216,5.706419,2,Unknown,NaN,Unknown
2,1,3272,5.671748,3,Fantasy,Romance,"Fantasy, Romance"
3,1,9050,5.649014,4,Unknown,NaN,Unknown
4,1,9063,5.623674,5,Children's,Classics,"Children's, Classics"
5,1,8924,5.592497,6,Children's,Mystery,"Children's, Mystery"
6,1,2907,5.559885,7,Children's,Adventure,"Children's, Adventure"
7,1,9219,5.529706,8,Nonfiction,Drama,"Nonfiction, Drama"
8,1,9818,5.509216,9,Unknown,NaN,Unknown
9,1,9151,5.458622,10,Children's,Horror,"Children's, Horror"


In [4]:
import pandas as pd

# Read your table (replace with your file path)
df = test

# Group by user and collect the list of recommended book_ids
user_books = df.groupby("user_id")["book_id"].apply(list).reset_index()

# Print results
for _, row in user_books.iterrows():
    print(f"user_id: {row['user_id']} → {row['book_id']}")


user_id: 1 → [9534, 7216, 3272, 9050, 9063, 8924, 2907, 9219, 9818, 9151, 798, 9210, 3151, 5303, 6395, 2039, 9228, 8235, 7878, 976, 7317, 4434, 6006, 7368, 1548]
user_id: 2 → [9017, 5629, 5102, 1924, 7687, 9529, 6231, 5996, 6141, 1498, 77, 4019, 2638, 1923, 2033, 8825, 1318, 3093, 2051, 9958, 3368, 6467, 5880, 3911, 353]
user_id: 3 → [9534, 5046, 9234, 9709, 7485, 3226, 6618, 6120, 9379, 4088, 4743, 8142, 8634, 7831, 6578, 6522, 2082, 7705, 114, 4805, 9220, 9223, 9724, 3278, 5017]
user_id: 4 → [5102, 2638, 7582, 6141, 7229, 9091, 9413, 4419, 6084, 4653, 4154, 464, 5731, 9529, 77, 248, 1338, 6110, 1475, 8925, 2473, 9246, 5629, 114, 3061]
user_id: 5 → [8927, 9228, 7216, 5549, 5580, 3272, 4962, 8978, 8913, 1788, 780, 3628, 4483, 976, 9050, 8182, 9569, 2907, 6920, 1010, 9566, 6361, 798, 7181, 4574]
user_id: 6 → [5102, 7885, 5996, 8825, 2638, 9529, 5629, 464, 4955, 2557, 5695, 3360, 6184, 9958, 231, 3061, 7118, 4495, 6329, 1959, 77, 7226, 7229, 7253, 2980]
user_id: 7 → [9123, 7450, 8927, 95

In [27]:
genre = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/SVD/Book/data/df_final_with_genres.csv")

In [30]:
import pandas as pd

# Load the dataframe
dataframe = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/SVD/Book/data/df_final.csv")

# Count unique book IDs
unique_book_count = dataframe['book_id'].nunique()

print("Number of unique book IDs:", unique_book_count)


Number of unique book IDs: 10000


In [31]:
import pandas as pd

# Load the dataframe
dataframe = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/SVD/Book/data/df_final_with_genres.csv")

# Count unique book IDs
unique_book_count = dataframe['book_id'].nunique()

print("Number of unique book IDs:", unique_book_count)


Number of unique book IDs: 10000


In [32]:
example_book_id = dataframe['book_id'].sample(1).iloc[0]
print("Example book ID:", example_book_id)


Example book ID: 481


In [37]:
genre_example = dataframe.loc[dataframe['book_id'] == example_book_id, 'genres'].dropna()

if not genre_example.empty:
    print(f"Genre for book_id {example_book_id}:", genre_example.iloc[0])
else:
    print(f"No genre found for book_id {example_book_id}")


Genre for book_id 481: Fantasy, Mystery


In [41]:
import pandas as pd

# Load your data
dataframe = pd.read_csv("/home/moshtasa/Research/phd-svd-recsys/SVD/Book/data/df_final_with_genres.csv")

# Count total unique books
total_unique_books = dataframe['book_id'].nunique()

# Get unique books that have no genre (genres is NaN)
books_without_genre = dataframe[dataframe['genres'].isna()]['book_id'].nunique()

print("Total unique books:", total_unique_books)
print("Unique books WITHOUT genre:", books_without_genre)
print("Percentage missing genre:", round(books_without_genre / total_unique_books * 100, 2), "%")


Total unique books: 10000
Unique books WITHOUT genre: 585
Percentage missing genre: 5.85 %


2,4,6

In [5]:
#!/usr/bin/env python3
# make_children_explanations_and_figs_1017.py
#
# Single-genre workflow for Children's only (pos=5, neg=1):
#   - Reads ORIGINAL_{K}recommendation.csv from SVD_Single_Injection/
#   - Finds injections under SVD_Single_Injection/5:
#         f_Children_s_{N}u_pos5_neg1_all_{K}recommendation.csv
#   - Computes per-dataset metrics + per-book rankings (within Children's)
#   - Saves:
#       <OUT_BASE>/children_s_explanation/explanation.txt
#       <OUT_BASE>/children_s_explanation/per_book_ranking.csv
#       <OUT_BASE>/children_s_explanation/children_s__pos5.png
#
# Python 3.8+

from pathlib import Path
from typing import Optional, Dict, Tuple, List
import re
import pandas as pd

import matplotlib
matplotlib.use("Agg")   # headless-safe
import matplotlib.pyplot as plt

# ========= CONFIG (edit paths if needed) =====================================
ROOT        = Path("/home/moshtasa/Research/phd-svd-recsys/SVD/Book/result/rec/top_re/1017")
ORIG_DIR    = ROOT / "SVD_Single_Injection"      # contains ORIGINAL_{K}recommendation.csv
SEARCH_ROOT = ROOT / "SVD_Single_Injection" / "5"  # where Children's injections live
OUT_BASE    = ROOT / "result" / "5"

K_LIST = [15, 25, 35]
N_LIST = [2, 4, 6, 25, 50, 100, 200, 300, 350, 500, 1000]
INJECTION_PATTERN = "f_*_{n}u_pos5_neg1_all_{k}recommendation.csv"

# ========= Genre normalization (Children's only, robust) =====================
def _clean_space_lower(s: str) -> str:
    s = re.sub(r"[_\s]+", " ", str(s).strip())
    s = s.replace("’", "'")
    return s.lower()

GENRE_NORM = {
    "childrens": "children's",
    "children s": "children's",
    "children": "children's",
    "children_s": "children's",
}
DISPLAY_MAP = {"children's": "Children's"}

def norm_token(s: str) -> str:
    s = _clean_space_lower(s)
    s_no_apos = s.replace("'", "")
    return GENRE_NORM.get(s_no_apos, s)

def display_canonical(s: str) -> str:
    low = norm_token(s)
    return DISPLAY_MAP.get(low, s)

def slugify_token(x: str) -> str:
    x = re.sub(r"[^A-Za-z0-9]+", "_", str(x)).strip("_").lower()
    return re.sub(r"_+", "_", x)

TARGET_CANON = "children's"
TARGET_DISP  = display_canonical(TARGET_CANON)

# ========= IO helpers ========================================================
def load_rec_csv(fp: Path) -> pd.DataFrame:
    df = pd.read_csv(fp, low_memory=False)
    if "genres_all" not in df.columns:
        df["genres_all"] = ""
    for c in ["user_id", "book_id", "rank", "est_score", "original_title"]:
        if c not in df.columns:
            df[c] = pd.NA
    return df

def has_genre(cell, target_canon_lc: str) -> bool:
    if pd.isna(cell):
        return False
    tokens = []
    for t in str(cell).split(","):
        t = t.strip()
        if not t:
            continue
        tokens.append(norm_token(t))
    return target_canon_lc in tokens

def metrics_for_file(df: pd.DataFrame, target_disp: str) -> Tuple[int, float, int, int, int, pd.Series]:
    unique_books_in_file = df["book_id"].nunique() if "book_id" in df.columns else 0
    target_canon = norm_token(target_disp)
    is_tg = df["genres_all"].apply(lambda s: has_genre(s, target_canon))

    if "user_id" in df.columns:
        per_user = df.assign(is_tg=is_tg).groupby("user_id")["is_tg"].sum()
        avg_per_user = float(per_user.mean()) if not per_user.empty else 0.0
        users_with_target = int((per_user > 0).sum())
    else:
        avg_per_user = 0.0
        users_with_target = 0

    unique_tg_books = df.loc[is_tg, "book_id"].nunique() if "book_id" in df.columns else 0
    freq = int(is_tg.sum())
    return unique_books_in_file, avg_per_user, unique_tg_books, freq, users_with_target, is_tg

def per_book_ranking(df_target_rows: pd.DataFrame) -> pd.DataFrame:
    if df_target_rows.empty:
        return pd.DataFrame(columns=[
            "book_id","rank","freq","users_n","avg_rank","avg_est_score","rank1_count","original_title","genres_all"
        ])

    g = df_target_rows.groupby("book_id", as_index=False)
    out = g.agg(
        freq=("book_id", "size"),
        users_n=("user_id", "nunique"),
        avg_rank=("rank", "mean"),
        avg_est_score=("est_score", "mean"),
        rank1_count=("rank", lambda s: int((s == 1).sum())),
    )
    for c in ["original_title", "genres_all"]:
        smpl = df_target_rows.groupby("book_id")[c].apply(
            lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA
        ).reset_index(name=c)
        out = out.merge(smpl, on="book_id", how="left")

    out = out.sort_values(["freq","rank1_count","avg_rank"], ascending=[False, False, True]).reset_index(drop=True)
    out.insert(1, "rank", out.index + 1)
    return out

def extract_genre_from_injection_filename(p: Path) -> Optional[str]:
    name = p.name
    m = re.match(r"^f_(.+?)_(\d+)u_pos\d+_neg[^_]+_all_(\d+)recommendation\.csv$", name, flags=re.IGNORECASE)
    if not m:
        return None
    raw_genre = m.group(1)
    return norm_token(raw_genre)

def find_children_injection(n: int, k: int) -> Optional[Path]:
    patt = INJECTION_PATTERN.format(n=n, k=k)
    for p in SEARCH_ROOT.rglob(patt):
        g = extract_genre_from_injection_filename(p)
        if g and g == TARGET_CANON:
            return p
    return None

# ========= Plotting ==========================================================
def plot_children_pos_three_bins(data_by_k: Dict[int, Dict[str, float]], out_png: Path):
    ks = sorted(data_by_k.keys())
    if not ks:
        return
    groups = ["Original"] + [str(x) for x in N_LIST]
    present_groups = [g for g in groups if any(g in data_by_k.get(k, {}) for k in ks)]

    width = 0.8 / max(1, len(present_groups))
    x = list(range(len(ks)))
    plt.figure(figsize=(8.4, 4.4), dpi=160)

    for j, g in enumerate(present_groups):
        offs = [i + (j - (len(present_groups)-1)/2)*width for i in x]
        vals = [float(data_by_k.get(k, {}).get(g, 0.0)) for k in ks]
        plt.bar(offs, vals, width=width, label=("n="+g if g!="Original" else "Original"))

    plt.xticks(x, [f"Top-{k}" for k in ks])
    plt.ylabel("Avg # of Children's books in Top-K per user")
    plt.title("Children's — POS=5")
    plt.legend(ncol=min(4, len(present_groups)), fontsize=9)
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png)
    plt.close()

# ========= MAIN (Children's only) ============================================
def main():
    GEN_KEY = "children_s"
    OUT_DIR = OUT_BASE / f"{GEN_KEY}_explanation"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    original_uniques: Dict[int, int] = {}
    injection_uniques: Dict[Tuple[int,int], int] = {}
    per_book_rows: List[Dict] = []
    data_by_k: Dict[int, Dict[str, float]] = {k: {} for k in K_LIST}

    # ORIGINALS
    for K in K_LIST:
        fp = ORIG_DIR / f"ORIGINAL_{K}recommendation.csv"
        if not fp.exists():
            print(f"[warn][Children's] Missing ORIGINAL for K={K}: {fp}")
            continue
        df = load_rec_csv(fp)
        uniq_all, avg_user, uniq_tg, freq, users_with_tg, is_tg = metrics_for_file(df, TARGET_DISP)
        print(f"[Children's] [{K}], {fp} -> {uniq_all} books -> Children's: avg/user={avg_user:.2f}, unique={uniq_tg}")
        original_uniques[K] = uniq_tg
        data_by_k[K]["Original"] = avg_user

        ranked = per_book_ranking(df.loc[is_tg].copy())
        for _, r in ranked.iterrows():
            per_book_rows.append({
                "genre": TARGET_DISP,
                "dataset": f"original{K}",
                "book_id": int(r["book_id"]) if pd.notna(r["book_id"]) else None,
                "rank": int(r["rank"]) if pd.notna(r["rank"]) else None,
                "freq": int(r["freq"]) if pd.notna(r["freq"]) else 0,
                "users_n": int(r["users_n"]) if pd.notna(r["users_n"]) else 0,
                "avg_rank": float(r["avg_rank"]) if pd.notna(r["avg_rank"]) else None,
                "avg_est_score": float(r["avg_est_score"]) if pd.notna(r["avg_est_score"]) else None,
                "rank1_count": int(r["rank1_count"]) if pd.notna(r["rank1_count"]) else 0,
                "original_title": r.get("original_title", pd.NA),
                "genres_all": r.get("genres_all", pd.NA),
            })

    # INJECTIONS
    for K in K_LIST:
        for N in N_LIST:
            fp = find_children_injection(N, K)
            if not fp:
                print(f"[warn][Children's] Missing injection file for N={N}, K={K}")
                continue
            df = load_rec_csv(fp)
            uniq_all, avg_user, uniq_tg, freq, users_with_tg, is_tg = metrics_for_file(df, TARGET_DISP)
            print(f"[Children's] [{K}] {fp.name} -> {uniq_all} books -> Children's: avg/user={avg_user:.2f}, unique={uniq_tg}")
            injection_uniques[(N, K)] = uniq_tg
            data_by_k[K][str(N)] = avg_user

            ranked = per_book_ranking(df.loc[is_tg].copy())
            for _, r in ranked.iterrows():
                per_book_rows.append({
                    "genre": TARGET_DISP,
                    "dataset": f"{N}u_{K}",
                    "book_id": int(r["book_id"]) if pd.notna(r["book_id"]) else None,
                    "rank": int(r["rank"]) if pd.notna(r["rank"]) else None,
                    "freq": int(r["freq"]) if pd.notna(r["freq"]) else 0,
                    "users_n": int(r["users_n"]) if pd.notna(r["users_n"]) else 0,
                    "avg_rank": float(r["avg_rank"]) if pd.notna(r["avg_rank"]) else None,
                    "avg_est_score": float(r["avg_est_score"]) if pd.notna(r["avg_est_score"]) else None,
                    "rank1_count": int(r["rank1_count"]) if pd.notna(r["rank1_count"]) else 0,
                    "original_title": r.get("original_title", pd.NA),
                    "genres_all": r.get("genres_all", pd.NA),
                })

    # Write summaries
    text_path = OUT_DIR / "explanation.txt"
    with open(text_path, "w", encoding="utf-8") as f:
        f.write(f"{TARGET_DISP.lower()}:\n")
        for K in K_LIST:
            if K in original_uniques:
                f.write(f"original {K}: number_of_unique_books: {original_uniques[K]}\n")
        for K in K_LIST:
            for N in N_LIST:
                val = injection_uniques.get((N, K))
                if val is not None:
                    f.write(f"{N}u, {K}, number_of_unique_books: {val}\n")

    table_df = pd.DataFrame(per_book_rows)
    table_df.to_csv(OUT_DIR / "per_book_ranking.csv", index=False)

    fig_path = OUT_DIR / f"{slugify_token(TARGET_DISP)}__pos5.png"
    plot_children_pos_three_bins(data_by_k, fig_path)

    print(f"[OK][Children's] Saved text:   {text_path}")
    print(f"[OK][Children's] Saved table:  {OUT_DIR / 'per_book_ranking.csv'}")
    print(f"[OK][Children's] Saved figure: {fig_path}")

if __name__ == "__main__":
    main()


[Children's] [15], /home/moshtasa/Research/phd-svd-recsys/SVD/Book/result/rec/top_re/1017/SVD_Single_Injection/ORIGINAL_15recommendation.csv -> 2041 books -> Children's: avg/user=1.07, unique=187
[Children's] [25], /home/moshtasa/Research/phd-svd-recsys/SVD/Book/result/rec/top_re/1017/SVD_Single_Injection/ORIGINAL_25recommendation.csv -> 2771 books -> Children's: avg/user=2.08, unique=268
[Children's] [35], /home/moshtasa/Research/phd-svd-recsys/SVD/Book/result/rec/top_re/1017/SVD_Single_Injection/ORIGINAL_35recommendation.csv -> 3366 books -> Children's: avg/user=3.15, unique=336
[Children's] [15] f_Children_s_2u_pos5_neg1_all_15recommendation.csv -> 2029 books -> Children's: avg/user=1.27, unique=198
[Children's] [15] f_Children_s_4u_pos5_neg1_all_15recommendation.csv -> 2201 books -> Children's: avg/user=1.81, unique=238
[Children's] [15] f_Children_s_6u_pos5_neg1_all_15recommendation.csv -> 1906 books -> Children's: avg/user=2.46, unique=218
[Children's] [15] f_Children_s_25u_pos5_

In [2]:
#!/usr/bin/env python3
# make_all_genres_explanations_and_figs_1016.py
#
# Merged workflow (single-genre mode, no pairs):
#  • For each of the 13 canonical genres:
#      - reads ORIGINAL_{K}recommendation.csv
#      - finds injection files like f_<Genre>_{N}u_pos5_neg1_all_{K}recommendation.csv
#      - computes per-dataset metrics + per-book rankings (within the target genre)
#      - saves:
#           <OUT_BASE>/<GEN_KEY>_explanation/explanation.txt
#           <OUT_BASE>/<GEN_KEY>_explanation/per_book_ranking.csv
#           <OUT_BASE>/<GEN_KEY>_explanation/<gen_slug>__pos5.png   ← NEW: figure per genre
#
#  • Global rollups:
#      <OUT_BASE>/_all_genres/summary_master.txt
#      <OUT_BASE>/_all_genres/per_book_ranking_all.csv
#
# Python 3.8+

from pathlib import Path
from typing import Optional, Dict, Tuple, List
import re
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ========= CONFIG (edit paths if needed) =====================================
ROOT        = Path("/home/moshtasa/Research/phd-svd-recsys/SVD/Book/result/rec/top_re/1017")
ORIG_DIR    = ROOT / "SVD_Single_Injection"         # ORIGINAL_{K}recommendation.csv live here
SEARCH_ROOT = ROOT                       # recurse to find f_<Genre>_{n}u_pos5_neg1_all_{K}recommendation.csv
OUT_BASE    = ROOT / "result" / "5"

K_LIST = [15, 25, 35]
N_LIST = [2, 4, 6, 25, 50, 100, 200, 300, 350, 500, 1000]   # synthetic user counts for injections

# Try patterns in order while searching for injections
INJECTION_PATTERNS = [
    "f_*_{n}u_pos5_neg1_all_{k}recommendation.csv",
    # "f_*_{n}u_pos5_neg0_all_{k}recommendation.csv",  # uncomment if you also use neg0 variants
]

# ========= Genre normalization ===============================================
CANONICAL_GENRES = [
    "Adventure", "Classics", "Drama", "Fantasy", "Historical", "Horror",
    "Mystery", "Nonfiction", "Romance", "Science Fiction", "Thriller",
    "Children's", "Adult",
]

GENRE_NORM: Dict[str, str] = {
    # canonical (lowercased)
    "adventure": "adventure",
    "classics": "classics",
    "drama": "drama",
    "fantasy": "fantasy",
    "historical": "historical",
    "horror": "horror",
    "mystery": "mystery",
    "nonfiction": "nonfiction",
    "romance": "romance",
    "science fiction": "science fiction",
    "thriller": "thriller",
    "children's": "children's",
    "adult": "adult",
    # filename/typo guards
    "children_s": "children's",
    "childrens": "children's",
    "children’s": "children's",
    "children": "children's",
    "sci-fi": "science fiction",
    "scifi": "science fiction",
    "science_finction": "science fiction",
    "adventre": "adventure",
    "advenbture": "adventure",
}

DISPLAY_MAP = {
    "adventure": "Adventure",
    "classics": "Classics",
    "drama": "Drama",
    "fantasy": "Fantasy",
    "historical": "Historical",
    "horror": "Horror",
    "mystery": "Mystery",
    "nonfiction": "Nonfiction",
    "romance": "Romance",
    "science fiction": "Science Fiction",
    "thriller": "Thriller",
    "children's": "Children's",
    "adult": "Adult",
}

def _clean_space_lower(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s).strip())
    return s.lower()

def norm_token(s: str) -> str:
    low = _clean_space_lower(s)
    return GENRE_NORM.get(low, low)

def display_canonical(s: str) -> str:
    low = norm_token(s)
    return DISPLAY_MAP.get(low, s)

def slugify_token(x: str) -> str:
    x = re.sub(r"[^A-Za-z0-9]+", "_", str(x)).strip("_").lower()
    return re.sub(r"_+", "_", x)

# ========= IO helpers ========================================================
def load_rec_csv(fp: Path) -> pd.DataFrame:
    df = pd.read_csv(fp, low_memory=False)
    # ensure presence of common columns
    if "genres_all" not in df.columns:
        df["genres_all"] = ""
    for c in ["user_id", "book_id", "rank", "est_score", "original_title"]:
        if c not in df.columns:
            df[c] = pd.NA
    return df

def has_genre(cell, target_canon_lc: str) -> bool:
    if pd.isna(cell):
        return False
    tokens = []
    for t in str(cell).split(","):
        t = t.strip()
        if not t:
            continue
        tokens.append(norm_token(t))
    return target_canon_lc in tokens

def metrics_for_file(df: pd.DataFrame, target_disp: str) -> Tuple[int, float, int, int, int, pd.Series]:
    """
    Returns:
      unique_books_in_file,
      avg_per_user (of target-genre books),
      unique_target_books,
      freq (total target-genre rows),
      users_with_target,
      is_target_mask
    """
    unique_books_in_file = df["book_id"].nunique() if "book_id" in df.columns else 0
    target_canon = norm_token(target_disp)
    is_tg = df["genres_all"].apply(lambda s: has_genre(s, target_canon))

    if "user_id" in df.columns:
        per_user = df.assign(is_tg=is_tg).groupby("user_id")["is_tg"].sum()
        avg_per_user = float(per_user.mean()) if not per_user.empty else 0.0
        users_with_target = int((per_user > 0).sum())
    else:
        avg_per_user = 0.0
        users_with_target = 0

    unique_tg_books = df.loc[is_tg, "book_id"].nunique() if "book_id" in df.columns else 0
    freq = int(is_tg.sum())
    return unique_books_in_file, avg_per_user, unique_tg_books, freq, users_with_target, is_tg

def per_book_ranking(df_target_rows: pd.DataFrame) -> pd.DataFrame:
    """
    Build per-book table from rows already filtered to target genre:
      rank by freq desc, rank1_count desc, avg_rank asc
    """
    if df_target_rows.empty:
        return pd.DataFrame(columns=[
            "book_id","rank","freq","users_n","avg_rank","avg_est_score","rank1_count","original_title","genres_all"
        ])

    g = df_target_rows.groupby("book_id", as_index=False)
    out = g.agg(
        freq=("book_id", "size"),
        users_n=("user_id", "nunique"),
        avg_rank=("rank", "mean"),
        avg_est_score=("est_score", "mean"),
        rank1_count=("rank", lambda s: int((s == 1).sum())),
    )
    # attach sample title/genres
    for c in ["original_title", "genres_all"]:
        smpl = df_target_rows.groupby("book_id")[c].apply(
            lambda s: s.dropna().iloc[0] if s.notna().any() else pd.NA
        ).reset_index(name=c)
        out = out.merge(smpl, on="book_id", how="left")

    out = out.sort_values(["freq","rank1_count","avg_rank"], ascending=[False, False, True]).reset_index(drop=True)
    out.insert(1, "rank", out.index + 1)
    return out

def extract_k(path: Path) -> Optional[int]:
    m = re.search(r"(\d+)\s*recommendation$", path.stem)
    return int(m.group(1)) if m else None

def extract_genre_from_injection_filename(p: Path) -> Optional[str]:
    """
    Accept names like:
      f_Adventure_2u_pos5_neg1_all_15recommendation.csv
      f_Science_Fiction_4u_pos5_neg1_all_35recommendation.csv
    """
    name = p.name
    m = re.match(r"^f_(.+?)_(\d+)u_pos\d+_neg[^_]+_all_(\d+)recommendation\.csv$", name, flags=re.IGNORECASE)
    if not m:
        return None
    raw_genre = m.group(1).replace("_", " ")
    return norm_token(raw_genre)

def find_injection(target_disp: str, n: int, k: int) -> Optional[Path]:
    """
    Search recursively for an injection file matching target genre, n users, and K.
    """
    target_canon = norm_token(target_disp)
    for pat in INJECTION_PATTERNS:
        patt = pat.format(n=n, k=k)
        for p in SEARCH_ROOT.rglob(patt):
            g = extract_genre_from_injection_filename(p)
            if g and g == target_canon:
                return p
    return None

def genre_key_disp(genre_name: str) -> Tuple[str, str]:
    disp = display_canonical(genre_name)
    key  = re.sub(r"[^\w\-]+", "_", disp)
    return key, disp

# ========= Plotting ==========================================================
def plot_genre_pos_three_bins(G_disp: str,
                              data_by_k: Dict[int, Dict[str, float]],
                              out_png: Path):
    """
    data_by_k: {K: {"Original": v0, "2": v2, "4": v4, "6": v6}}
    """
    ks = sorted(data_by_k.keys())
    if not ks:
        return
    groups = ["Original"] + [str(x) for x in N_LIST]
    # include only groups that appear at least once
    present_groups = [g for g in groups if any(g in data_by_k.get(k, {}) for k in ks)]

    width = 0.8 / max(1, len(present_groups))
    x = list(range(len(ks)))
    plt.figure(figsize=(8.4, 4.4), dpi=160)

    for j, g in enumerate(present_groups):
        offs = [i + (j - (len(present_groups)-1)/2)*width for i in x]
        vals = [float(data_by_k.get(k, {}).get(g, 0.0)) for k in ks]
        plt.bar(offs, vals, width=width, label=("n="+g if g!="Original" else "Original"))

    plt.xticks(x, [f"Top-{k}" for k in ks])
    plt.ylabel("Avg # of genre-books in Top-K per user")
    plt.title(f"{G_disp} — POS=5")
    plt.legend(ncol=min(4, len(present_groups)), fontsize=9)
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png)
    plt.close()

# ========= MAIN per-genre ====================================================
def process_one_genre(target_genre: str) -> Dict:
    """
    Processes one genre; returns dict with:
      {
        'disp': display name,
        'original_uniques': {K: unique_tg_books, ...},
        'injection_uniques': {(N,K): unique_tg_books, ...},
        'out_dir': Path,
        'per_book_rows': [ ... rows with 'genre' ... ]
      }
    Also writes the figure into the same OUT_DIR.
    """
    GEN_KEY, GEN_DISP = genre_key_disp(target_genre)
    OUT_DIR = OUT_BASE / f"{GEN_KEY}_explanation"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    original_uniques: Dict[int, int] = {}
    injection_uniques: Dict[Tuple[int,int], int] = {}
    per_book_rows: List[Dict] = []

    # Figure build-up dict
    data_by_k: Dict[int, Dict[str, float]] = {k: {} for k in K_LIST}

    # ORIGINAL
    for K in K_LIST:
        fp = ORIG_DIR / f"ORIGINAL_{K}recommendation.csv"
        if not fp.exists():
            print(f"[warn][{GEN_DISP}] Missing ORIGINAL for K={K}: {fp}")
            continue
        df = load_rec_csv(fp)
        uniq_all, avg_user, uniq_tg, freq, users_with_tg, is_tg = metrics_for_file(df, GEN_DISP)
        print(f"[{GEN_DISP}] [{K}], {fp} -> {uniq_all} books -> {GEN_DISP}: avg/user={avg_user:.2f}, unique={uniq_tg}")
        original_uniques[K] = uniq_tg
        data_by_k[K]["Original"] = avg_user

        ranked = per_book_ranking(df.loc[is_tg].copy())
        for _, r in ranked.iterrows():
            per_book_rows.append({
                "genre": GEN_DISP,
                "dataset": f"original{K}",
                "book_id": int(r["book_id"]),
                "rank": int(r["rank"]),
                "freq": int(r["freq"]),
                "users_n": int(r["users_n"]),
                "avg_rank": float(r["avg_rank"]) if pd.notna(r["avg_rank"]) else None,
                "avg_est_score": float(r["avg_est_score"]) if pd.notna(r["avg_est_score"]) else None,
                "rank1_count": int(r["rank1_count"]),
                "original_title": r.get("original_title", pd.NA),
                "genres_all": r.get("genres_all", pd.NA),
            })

    # INJECTIONS
    for K in K_LIST:
        for N in N_LIST:
            fp = find_injection(GEN_DISP, N, K)
            if not fp:
                print(f"[warn][{GEN_DISP}] Missing injection file for N={N}, K={K}")
                continue
            df = load_rec_csv(fp)
            uniq_all, avg_user, uniq_tg, freq, users_with_tg, is_tg = metrics_for_file(df, GEN_DISP)
            print(f"[{GEN_DISP}] [{K}] {fp.name} -> {uniq_all} books -> {GEN_DISP}: avg/user={avg_user:.2f}, unique={uniq_tg}")
            injection_uniques[(N, K)] = uniq_tg
            data_by_k[K][str(N)] = avg_user

            ranked = per_book_ranking(df.loc[is_tg].copy())
            for _, r in ranked.iterrows():
                per_book_rows.append({
                    "genre": GEN_DISP,
                    "dataset": f"{N}u_{K}",
                    "book_id": int(r["book_id"]),
                    "rank": int(r["rank"]),
                    "freq": int(r["freq"]),
                    "users_n": int(r["users_n"]),
                    "avg_rank": float(r["avg_rank"]) if pd.notna(r["avg_rank"]) else None,
                    "avg_est_score": float(r["avg_est_score"]) if pd.notna(r["avg_est_score"]) else None,
                    "rank1_count": int(r["rank1_count"]),
                    "original_title": r.get("original_title", pd.NA),
                    "genres_all": r.get("genres_all", pd.NA),
                })

    # Write per-genre text
    text_path = OUT_DIR / "explanation.txt"
    with open(text_path, "w", encoding="utf-8") as f:
        f.write(f"{GEN_DISP.lower()}:\n")
        for K in K_LIST:
            if K in original_uniques:
                f.write(f"original {K}: number_of_unique_books: {original_uniques[K]}\n")
        for K in K_LIST:
            for N in N_LIST:
                val = injection_uniques.get((N, K))
                if val is not None:
                    f.write(f"{N}u, {K}, number_of_unique_books: {val}\n")

    # Write per-genre ranking table
    table_df = pd.DataFrame(per_book_rows)
    table_path = OUT_DIR / "per_book_ranking.csv"
    table_df.to_csv(table_path, index=False)

    # Make and save per-genre figure into the same folder
    fig_path = OUT_DIR / f"{slugify_token(GEN_DISP)}__pos5.png"
    plot_genre_pos_three_bins(GEN_DISP, data_by_k, fig_path)

    print(f"[OK][{GEN_DISP}] Saved text:   {text_path}")
    print(f"[OK][{GEN_DISP}] Saved table:  {table_path}")
    print(f"[OK][{GEN_DISP}] Saved figure: {fig_path}")

    return {
        "disp": GEN_DISP,
        "original_uniques": original_uniques,
        "injection_uniques": injection_uniques,
        "out_dir": OUT_DIR,
        "per_book_rows": per_book_rows,
    }

# ========= MAIN (all genres) =================================================
def main():
    OUT_ALL = OUT_BASE / "_all_genres"
    OUT_ALL.mkdir(parents=True, exist_ok=True)

    all_rows: List[Dict] = []
    master_lines: List[str] = []

    for g in CANONICAL_GENRES:
        res = process_one_genre(g)
        # Append master block
        master_lines.append(f"{res['disp'].lower()}:")
        for K in K_LIST:
            if K in res["original_uniques"]:
                master_lines.append(f"original {K}: number_of_unique_books: {res['original_uniques'][K]}")
        for K in K_LIST:
            for N in N_LIST:
                val = res["injection_uniques"].get((N, K))
                if val is not None:
                    master_lines.append(f"{N}u, {K}, number_of_unique_books: {val}")
        master_lines.append("")  # blank line
        all_rows.extend(res["per_book_rows"])

    # Save global rollups
    master_txt = OUT_ALL / "summary_master.txt"
    with open(master_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(master_lines))
    print(f"[OK] Wrote master summary: {master_txt}")

    if all_rows:
        df_all = pd.DataFrame(all_rows)
        df_all.to_csv(OUT_ALL / "per_book_ranking_all.csv", index=False)
        print(f"[OK] Wrote global per-book ranking CSV: {OUT_ALL / 'per_book_ranking_all.csv'}")
    else:
        print("[warn] No per-book rows produced. Check inputs / paths.")

if __name__ == "__main__":
    main()


[Adventure] [15], /home/moshtasa/Research/phd-svd-recsys/SVD/Book/result/rec/top_re/1017/SVD_Single_Injection/ORIGINAL_15recommendation.csv -> 2041 books -> Adventure: avg/user=2.87, unique=398
[Adventure] [25], /home/moshtasa/Research/phd-svd-recsys/SVD/Book/result/rec/top_re/1017/SVD_Single_Injection/ORIGINAL_25recommendation.csv -> 2771 books -> Adventure: avg/user=4.90, unique=549
[Adventure] [35], /home/moshtasa/Research/phd-svd-recsys/SVD/Book/result/rec/top_re/1017/SVD_Single_Injection/ORIGINAL_35recommendation.csv -> 3366 books -> Adventure: avg/user=6.99, unique=663
[Adventure] [15] f_Adventure_2u_pos5_neg1_all_15recommendation.csv -> 2044 books -> Adventure: avg/user=2.84, unique=401
[Adventure] [15] f_Adventure_4u_pos5_neg1_all_15recommendation.csv -> 2217 books -> Adventure: avg/user=3.23, unique=464
[Adventure] [15] f_Adventure_6u_pos5_neg1_all_15recommendation.csv -> 1979 books -> Adventure: avg/user=3.62, unique=435
[Adventure] [15] f_Adventure_25u_pos5_neg1_all_15recomm